# Task 2 - Advanced EDA + Text Mining of field failures
**Flow:** load -> profile -> clean (logged) -> detect free-text -> extract tags -> categorise issues -> cluster/topic-model -> trends -> root-cause screen -> export.
Put the assessment Excel in `data/raw/`; with no file present the notebook uses a SYNTHETIC demo set.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
import pandas as pd, matplotlib.pyplot as plt
from task2_field_failure_intelligence.src import eda, textmining as tm, topics
from task2_field_failure_intelligence.src.sample_data import make_failures
from task2_field_failure_intelligence.src.run_pipeline import RAW, PROC, FIG, find_cols, trends
files = [f for f in RAW.glob("*") if f.suffix.lower() in (".xlsx", ".xls", ".csv") and "SYNTHETIC" not in f.name]
demo = not files
raw = make_failures() if demo else eda.load_any(files[0])
print("DEMO DATA" if demo else files[0].name, raw.shape)

## 1. EDA - types, volume, per-column characteristics

In [ ]:
prof = eda.profile(raw); prof          # dtype, missing %, cardinality, sample, automatic free-text flag

In [ ]:
# Cleaning is a function that RETURNS A LOG, so every fix is auditable in the report.
d, log = eda.clean(raw); print(log)
eda.plot_eda(d, FIG)                   # distributions of numeric KPIs + counts of categorical fields (saved to reports/figures)
d.describe().round(2)

## 2. Text mining - unstructured -> structured

In [ ]:
date_col, text_cols, num_cols = find_cols(d); print("free text:", text_cols, "| date:", date_col)
print("vocabulary the dictionaries may be missing:", tm.expand_lexicon(d[text_cols[0]]))
m = tm.mine(d, text_cols)             # adds components / symptoms / actions / conditions / part_numbers / error_codes / issue_category
m["primary_condition"] = m["conditions"].map(lambda l: l[0] if l else "Unknown")
m[text_cols + ["components_str", "symptoms_str", "actions_str", "conditions_str", "issue_category"]].head(8)

In [ ]:
# Extraction coverage = how much of the free text we managed to structure (report this honestly)
print("rows with a component:", (m.primary_component != "Unknown").mean().round(3), "| with a symptom:", (m.primary_symptom != "Unknown").mean().round(3))
m["issue_category"].value_counts()

## 3. Patterns, clusters, topics

In [ ]:
piv, chg = trends(m, date_col); piv.plot(figsize=(9, 4), marker="o", ms=3); plt.title("Monthly failures by category"); plt.show(); chg

In [ ]:
m["cluster_id"], k, ctop, sil = topics.cluster_failures(m, text_cols); print("k =", k, "silhouette =", round(sil, 3)); print(ctop)
m["nmf_topic"], ntop = topics.nmf_topics(m, text_cols); print(ntop)

## 4. Root-cause screening
Lift = how over-represented an outcome is inside a driver group. `actionable` = lift >= 1.3 AND chi-square p < 0.05. This shows **association**, not proof - hand candidates to engineering for confirmation.

In [ ]:
drv = [c for c in ("product_model", "region", "primary_condition") if c in m]
rc = pd.concat([topics.root_cause_screen(m, outcome=o, drivers=drv) for o in ("primary_component", "issue_category")]).sort_values("lift", ascending=False)
rc.head(12)

In [ ]:
tag = "_SYNTHETIC_DEMO" if demo else ""
keep = [c for c in m.columns if c.endswith("_str")] + ["primary_component", "primary_symptom", "primary_condition", "issue_category", "cluster_id", "nmf_topic"]
m[list(d.columns) + keep].loc[:, lambda x: ~x.columns.duplicated()].to_excel(PROC / f"notebook_tags{tag}.xlsx", index=False)   # deliverable 2